<a href="https://colab.research.google.com/github/replyeshab/Mindsense-AI-Mental-Well-Being-app-/blob/main/MIndsenseAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q \
    openai \
    pydantic \
    sentence-transformers \
    transformers \
    datasets \
    scikit-learn \
    pandas \
    pytest

In [2]:
!wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
!wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
!wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv

--2026-10-20 16:22:49--  https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.118.207, 142.250.4.207, 64.233.170.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.118.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14174600 (14M) [application/octet-stream]
Saving to: ‘data/full_dataset/goemotions_1.csv’

goemotions_1.csv    100%[===================>]  13.52M  6.81MB/s    in 2.0s    

2026-10-20 16:22:52 (6.81 MB/s) - ‘data/full_dataset/goemotions_1.csv’ saved [14174600/14174600]

--2026-10-20 16:22:52--  https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.118.207, 142.250.4.207, 64.233.170.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.118.207|:443... connected.
HTTP request sent, awaiting res

In [3]:
%pip install -q pypdf sentence-transformers faiss-cpu pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.7 MB/s eta 0:00:00


In [4]:
%pip install -q -U openai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 570.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 11.9 MB/s eta 0:00:00


In [5]:
%pip install -U transformers accelerate bitsandbytes safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.1 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.5.1
    Uninstalling hf-xet-1.5.1:
      Successfully uninstalled hf-xet-1.5.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting

In [6]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen3-4B"

# Check GPU availability
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab, open "
        "Runtime > Change runtime type > select a GPU, "
        "then rerun this cell."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

# Select a supported compute precision
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

# Reduce model memory usage using 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading Qwen3-4B in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=compute_dtype,
    low_cpu_mem_usage=True,
)

model.eval()

print("✅ MindSense AI text model loaded successfully!")
print("Model device map:", model.hf_device_map)
print(
    "Model memory footprint:",
    round(model.get_memory_footprint() / 1024**3, 2),
    "GB"
)

GPU: Tesla T4
GPU memory: 14.56 GB
Loading tokenizer...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ MindSense AI text model loaded successfully!


AttributeError: 'Qwen3ForCausalLM' object has no attribute 'hf_device_map'

In [7]:
import json

SYSTEM_PROMPT = """
You are the text emotion analysis component of MindSense AI,
an emotional wellness and self-reflection application.

Analyze the user's message without diagnosing them.

Distinguish explicitly stated feelings from inferred feelings.
Consider context, sarcasm, ambiguity and mixed emotions.
Never claim to know someone's internal state with certainty.
Evidence must be an exact short phrase from the message.
If the evidence is weak, express high uncertainty.
Ask a gentle clarification question when useful.

Return ONLY valid JSON with this schema:
{
  "user_stated_feeling": "string",
  "signals": [
    {"emotion": "string", "evidence": "string"}
  ],
  "uncertainty": "low, medium, or high",
  "needs_clarification": true,
  "clarification_question": "string",
  "explanation": "string"
}
"""

def analyze_text_emotion(message):
    if not isinstance(message, str) or not message.strip():
        raise ValueError("Please provide a non-empty message.")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": message.strip()}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {
            "parse_error": True,
            "raw_response": response
        }

In [9]:
import sqlite3
import uuid
import json

from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone

DB_PATH = Path("/content/mindsense_ai.db")


def utc_now():
    return datetime.now(timezone.utc).isoformat()


@contextmanager
def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")

    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def init_database():
    with get_db() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS users (
            id TEXT PRIMARY KEY,
            created_at TEXT NOT NULL,
            status TEXT NOT NULL DEFAULT 'active'
                CHECK(status IN ('active', 'disabled'))
        );

        -- Every grant or withdrawal creates a separate event.
        CREATE TABLE IF NOT EXISTS consent_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT NOT NULL,
            purpose TEXT NOT NULL CHECK(
                purpose IN (
                    'personal_memory',
                    'model_improvement'
                )
            ),
            granted INTEGER NOT NULL CHECK(granted IN (0, 1)),
            policy_version TEXT NOT NULL,
            changed_at TEXT NOT NULL,
            FOREIGN KEY(user_id) REFERENCES users(id)
                ON DELETE CASCADE
        );

        CREATE TABLE IF NOT EXISTS user_memories (
            id TEXT PRIMARY KEY,
            user_id TEXT NOT NULL,
            memory_type TEXT NOT NULL CHECK(
                memory_type IN (
                    'response_preference',
                    'helpful_strategy',
                    'avoid_strategy',
                    'user_correction'
                )
            ),
            content TEXT NOT NULL,
            user_confirmed INTEGER NOT NULL DEFAULT 1
                CHECK(user_confirmed IN (0, 1)),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL,
            FOREIGN KEY(user_id) REFERENCES users(id)
                ON DELETE CASCADE
        );

        CREATE INDEX IF NOT EXISTS idx_memories_user
            ON user_memories(user_id);

        CREATE TABLE IF NOT EXISTS feedback_events (
            id TEXT PRIMARY KEY,
            user_id TEXT NOT NULL,
            rating TEXT NOT NULL CHECK(
                rating IN ('helpful', 'not_helpful', 'unsure')
            ),
            comment TEXT NOT NULL DEFAULT '',
            created_at TEXT NOT NULL,
            improvement_consent_event_id INTEGER,
            FOREIGN KEY(user_id) REFERENCES users(id)
                ON DELETE CASCADE,
            FOREIGN KEY(improvement_consent_event_id)
                REFERENCES consent_events(id) ON DELETE SET NULL
        );

        CREATE INDEX IF NOT EXISTS idx_feedback_user
            ON feedback_events(user_id);

        -- Use reviewed examples, not automatically harvested chats.
        CREATE TABLE IF NOT EXISTS evaluation_cases (
            id TEXT PRIMARY KEY,
            message TEXT NOT NULL,
            expected_label TEXT NOT NULL,
            split TEXT NOT NULL CHECK(
                split IN ('train', 'validation', 'test')
            ),
            review_status TEXT NOT NULL DEFAULT 'pending'
                CHECK(
                    review_status IN ('pending', 'approved', 'rejected')
                ),
            created_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS model_versions (
            id TEXT PRIMARY KEY,
            model_name TEXT NOT NULL,
            version TEXT NOT NULL,
            status TEXT NOT NULL CHECK(
                status IN ('candidate', 'current', 'retired')
            )
        );
        """)
